In [ ]:
##Basic QC --- 

In [ ]:
#1.Data Validation

In [ ]:
#1.1 File-level validation and #1.2 Column-level validation

In [19]:
import pandas as pd
import os
import glob

# ==========================================
# CONFIGURATION
# ==========================================
# Use relative path for GitHub portability
# This looks for a folder named 'data' in the same directory as the script
INPUT_FOLDER = "data" 
TARGET_YEAR = 2022 
# ==========================================

def validate_rainfall_folder(folder_path, target_year):
    REQUIRED_HEADERS = [
        'station_id', 'station_name', 'lat', 'lon', 'agency_name', 
        'province', 'amphoe', 'tumbon', 'date', 'rain(mm)'
    ]
    
    # Get the absolute path relative to this script
    current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
    target_path = os.path.join(current_dir, folder_path)
    
    files = glob.glob(os.path.join(target_path, "*.csv"))
    
    if not files:
        print(f"[-] No CSV files found in: {target_path}")
        return

    print(f"[+] Total files found: {len(files)}")
    print(f"[+] Target Validation Year: {target_year}\n")

    passed_count = 0
    failed_count = 0

    for file_path in files:
        file_name = os.path.basename(file_path)
        is_file_ok = True
        
        # --- Part 1: File-level Validation ---
        try:
            df = pd.read_csv(file_path, encoding='utf-8', sep=',')
            actual_headers = df.columns.tolist()
            
            if not all(col in actual_headers for col in REQUIRED_HEADERS):
                is_file_ok = False
                missing = [col for col in REQUIRED_HEADERS if col not in actual_headers]
                print(f"[x] {file_name}: Missing headers {missing}")
            
            if len(actual_headers) != len(set(actual_headers)):
                is_file_ok = False
                print(f"[x] {file_name}: Duplicate column names")

        except Exception as e:
            is_file_ok = False
            print(f"[x] {file_name}: Error reading file ({e})")

        # --- Part 2: Column-level Validation ---
        if is_file_ok:
            errors = []
            
            # 1. station_id syntax
            df['station_id'] = df['station_id'].astype(str).str.strip()
            if not df['station_id'].str.match(r'^[0-9]+$').all():
                errors.append("station_id (invalid syntax)")

            # 2. date range
            try:
                df['date'] = pd.to_datetime(df['date'])
                start_date = f"{target_year}-01-01"
                end_date = f"{target_year}-12-31"
                if not df['date'].between(start_date, end_date).all():
                    errors.append(f"date (outside {target_year})")
            except:
                errors.append("date (format error)")

            # 3. lat/lon & Thailand bounding box
            for col in ['lat', 'lon']:
                if not pd.api.types.is_numeric_dtype(df[col]):
                    errors.append(f"{col} (non-numeric)")
                else:
                    if col == 'lat' and not df['lat'].between(5.0, 21.5).all():
                        errors.append("lat (outside Thailand)")
                    if col == 'lon' and not df['lon'].between(97.0, 106.0).all():
                        errors.append("lon (outside Thailand)")

            # 4. rain(mm) numeric check
            if not pd.api.types.is_numeric_dtype(df['rain(mm)']):
                errors.append("rain(mm) (non-numeric)")

            if errors:
                is_file_ok = False
                print(f"[x] {file_name}: {', '.join(errors)}")
            else:
                print(f"[o] {file_name}: Passed")

        if is_file_ok:
            passed_count += 1
        else:
            failed_count += 1

    print("\n" + "="*40)
    print("VALIDATION SUMMARY")
    print(f"PASSED: {passed_count}")
    print(f"FAILED: {failed_count}")
    print("="*40)

if __name__ == "__main__":
    validate_rainfall_folder(INPUT_FOLDER, TARGET_YEAR)

[+] Total files found: 364
[+] Target Validation Year: 2022

[o] 2022-01-01.csv: Passed
[o] 2022-01-02.csv: Passed
[o] 2022-01-03.csv: Passed
[o] 2022-01-04.csv: Passed
[o] 2022-01-05.csv: Passed
[o] 2022-01-06.csv: Passed
[o] 2022-01-07.csv: Passed
[o] 2022-01-08.csv: Passed
[o] 2022-01-09.csv: Passed
[o] 2022-01-10.csv: Passed
[o] 2022-01-11.csv: Passed
[o] 2022-01-12.csv: Passed
[o] 2022-01-13.csv: Passed
[o] 2022-01-14.csv: Passed
[o] 2022-01-15.csv: Passed
[o] 2022-01-16.csv: Passed
[o] 2022-01-17.csv: Passed
[o] 2022-01-18.csv: Passed
[o] 2022-01-19.csv: Passed
[o] 2022-01-20.csv: Passed
[o] 2022-01-21.csv: Passed
[o] 2022-01-22.csv: Passed
[o] 2022-01-23.csv: Passed
[o] 2022-01-24.csv: Passed
[o] 2022-01-25.csv: Passed
[o] 2022-01-26.csv: Passed
[o] 2022-01-27.csv: Passed
[o] 2022-01-28.csv: Passed
[o] 2022-01-29.csv: Passed
[o] 2022-01-30.csv: Passed
[o] 2022-01-31.csv: Passed
[o] 2022-02-01.csv: Passed
[o] 2022-02-02.csv: Passed
[o] 2022-02-03.csv: Passed
[o] 2022-02-04.csv: P

In [ ]:
#2.Data Verification___Duplicate Check

In [ ]:
#2.1 Duplicate station codes and dates

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# ==================== CONFIGURATION ====================
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()

# Input: Files from previous "Repeated Values" step
INPUT_GLOB = os.path.join(BASE_DIR, "data", "repeated_passed", "*.csv")

# Output: Reports
REPORT_DIR = os.path.join(BASE_DIR, "output", "duplicate_check")
os.makedirs(REPORT_DIR, exist_ok=True)
OUT_SUMMARY = os.path.join(REPORT_DIR, "duplicate_removal_summary.csv")

# Output: Final Cleaned Data
FINAL_DIR = os.path.join(BASE_DIR, "data_final_qc")
os.makedirs(FINAL_DIR, exist_ok=True)

# Define Key Columns
ID_COL   = "station_id"
DATE_COL = "date"
RAIN_COL = "rain(mm)"
# =======================================================

files = sorted(glob.glob(INPUT_GLOB))
if not files:
    print(f"[-] No files found in: {INPUT_GLOB}")
    exit()

print(f"[+] Starting Duplicate Verification for {len(files)} files...")

summary_rows = []

for f in files:
    file_name = os.path.basename(f)
    try:
        df = pd.read_csv(f)
        
        # Check required columns
        if not {ID_COL, DATE_COL, RAIN_COL}.issubset(df.columns):
            print(f"[!] Skipping {file_name}: Missing required columns")
            continue

        initial_count = len(df)
        
        # --- Condition 1: Exact Duplicate (ID + Date + Rain are identical) ---
        # We handle this by using drop_duplicates on all three columns
        df_cleaned = df.drop_duplicates(subset=[ID_COL, DATE_COL, RAIN_COL], keep='first')
        exact_dup_count = initial_count - len(df_cleaned)

        # --- Condition 2: Duplicate with Conflict (ID + Date identical, but Rain differs) ---
        # Group by ID and Date, then calculate mean for Rain
        # Other columns (station_name, lat, lon, etc.) will take the first occurrence
        
        # Define how to aggregate each column
        agg_dict = {}
        for col in df_cleaned.columns:
            if col == RAIN_COL:
                agg_dict[col] = 'mean'  # Condition 2: Average the conflicting rain values
            elif col in [ID_COL, DATE_COL]:
                continue # These are our grouping keys
            else:
                agg_dict[col] = 'first' # Keep other metadata like station_name, lat, lon

        # Apply aggregation
        df_final = df_cleaned.groupby([ID_COL, DATE_COL], as_index=False).agg(agg_dict)
        
        conflict_dup_count = len(df_cleaned) - len(df_final)

        # Save the final cleaned file
        df_final.to_csv(os.path.join(FINAL_DIR, file_name), index=False)

        # Log summary for this file
        summary_rows.append({
            "source_file": file_name,
            "original_rows": initial_count,
            "exact_duplicates_removed": exact_dup_count,
            "conflicts_averaged": conflict_dup_count,
            "final_rows": len(df_final)
        })

        if exact_dup_count > 0 or conflict_dup_count > 0:
            print(f"[o] {file_name}: Removed {exact_dup_count} exact dups and resolved {conflict_dup_count} conflicts.")
        else:
            print(f"[o] {file_name}: Clean (No duplicates found)")

    except Exception as e:
        print(f"[!] Error processing {file_name}: {e}")

# Save overall summary report
pd.DataFrame(summary_rows).to_csv(OUT_SUMMARY, index=False)

print(f"\n[+] Duplicate Verification Completed.")
print(f"    - Summary Report: {OUT_SUMMARY}")
print(f"    - Final Cleaned Data: {FINAL_DIR}/")

In [ ]:
#2.2 Check station coordinates

In [11]:
import pandas as pd
import numpy as np
import re
import glob
import os
from math import radians, sin, cos, asin, sqrt

# ==================== CONFIGURATION ====================
# Using relative paths for GitHub portability
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()

INPUT_FOLDER = os.path.join(BASE_DIR, "data", "raw")
OUTPUT_FOLDER = os.path.join(BASE_DIR, "output")

# Create output folder if it doesn't exist
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

# Output filenames
OUT_REPORT_STATION = os.path.join(OUTPUT_FOLDER, "station_coordinate_qc_report.csv")
OUT_REPORT_DETAILS = os.path.join(OUTPUT_FOLDER, "station_coordinate_qc_details.csv")
OUT_DUP_PAIRS      = os.path.join(OUTPUT_FOLDER, "station_duplicate_pairs.csv")

# Thailand Bounding Box
BBOX = {"lat_min": 5.0, "lat_max": 21.5, "lon_min": 97.0, "lon_max": 106.0}

# Distance threshold for near-duplicates (meters)
NEAR_DUP_THRESHOLD_M = 100
# =======================================================

# ----------------- Utilities -----------------
def haversine(lat1, lon1, lat2, lon2):
    """Calculate the great circle distance between two points (meters)"""
    R = 6371000.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    c = 2 * asin(sqrt(a))
    return R * c

def looks_like_dms(s):
    """Detect DMS symbols (°, ′, ″, ', ") or basic DMS patterns"""
    if pd.isna(s): return False
    return bool(re.search(r"[°′'″\"]", str(s)))

def to_numeric_or_nan(x):
    return pd.to_numeric(x, errors="coerce")

def mode_or_first(series):
    s = series.dropna()
    if s.empty: return np.nan
    try:
        return s.mode().iloc[0]
    except Exception:
        return s.iloc[0]

# ----------------- Load Data -----------------
search_path = os.path.join(INPUT_FOLDER, "*.csv")
files = sorted(glob.glob(search_path))

if not files:
    print(f"[-] Warning: No CSV files found in {INPUT_FOLDER}")
    # For GitHub, we don't want to crash, just exit gracefully
    exit()

dfs = []
for f in files:
    try:
        df = pd.read_csv(f)
        # Standardize column names (trim spaces)
        df.columns = df.columns.str.strip()
        
        required = {"station_id", "lat", "lon"} # Minimal required
        if not required.issubset(df.columns):
            print(f"[!] Skipping {os.path.basename(f)}: Missing columns {required - set(df.columns)}")
            continue

        df["_source_file"] = os.path.basename(f)
        dfs.append(df)
    except Exception as e:
        print(f"[!] Error reading {f}: {e}")

if not dfs:
    print("[-] No valid data to process.")
    exit()

all_df = pd.concat(dfs, ignore_index=True)

# Preserve raw values for DMS check
all_df["_raw_lat"] = all_df["lat"].astype(str)
all_df["_raw_lon"] = all_df["lon"].astype(str)

# Convert to numeric for range checks
all_df["_lat"] = to_numeric_or_nan(all_df["lat"])
all_df["_lon"] = to_numeric_or_nan(all_df["lon"])

# ----------------- Step 1–3: Record-level Validation -----------------
detail_rows = []

for idx, r in all_df.iterrows():
    sid = r.get("station_id", "N/A")
    sname = r.get("station_name_en", r.get("station_name_th", "Unknown"))
    src = r["_source_file"]
    lat = r["_lat"]
    lon = r["_lon"]
    raw_lat = r["_raw_lat"]
    raw_lon = r["_raw_lon"]

    # Step 1: Range Validation
    if pd.isna(lat) or pd.isna(lon):
        detail_rows.append([sid, sname, src, "Step1", "missing_lat_or_lon", raw_lat, raw_lon])
    else:
        if not (-90 <= lat <= 90):
            detail_rows.append([sid, sname, src, "Step1", "lat_out_of_range", lat, lon])
        if not (-180 <= lon <= 180):
            detail_rows.append([sid, sname, src, "Step1", "lon_out_of_range", lat, lon])
        if abs(lat) < 1e-12 and abs(lon) < 1e-12:
            detail_rows.append([sid, sname, src, "Step1", "lat_lon_is_zero", lat, lon])

    # Step 2: Thailand Bounding Box
    if not pd.isna(lat) and not pd.isna(lon):
        in_bbox = (BBOX["lat_min"] <= lat <= BBOX["lat_max"]) and (BBOX["lon_min"] <= lon <= BBOX["lon_max"])
        if not in_bbox:
            detail_rows.append([sid, sname, src, "Step2", "outside_thailand_bbox", lat, lon])

    # Step 3: Coordinate System Consistency
    if looks_like_dms(raw_lat) or looks_like_dms(raw_lon):
        detail_rows.append([sid, sname, src, "Step3", "contains_dms_symbols", raw_lat, raw_lon])

    if not pd.isna(lat) and not pd.isna(lon):
        if abs(lat) > 200 or abs(lon) > 200:
            detail_rows.append([sid, sname, src, "Step3", "magnitude_implausible", lat, lon])

        # Swap Check
        lat_like_lon = (BBOX["lon_min"] <= lat <= BBOX["lon_max"])
        lon_like_lat = (BBOX["lat_min"] <= lon <= BBOX["lat_max"])
        if lat_like_lon and lon_like_lat:
            detail_rows.append([sid, sname, src, "Step3", "suspect_lat_lon_swap", lat, lon])

# Save detailed report
details_cols = ["station_id", "station_name", "source_file", "step", "issue", "lat_value", "lon_value"]
details_df = pd.DataFrame(detail_rows, columns=details_cols)
details_df.to_csv(OUT_REPORT_DETAILS, index=False)

# ----------------- Step 4: Spatial Duplicates -----------------
# Get reference coordinate per station using Mode
station_ref = (all_df.groupby("station_id", dropna=False)
               .agg(lat_ref=("_lat", mode_or_first),
                    lon_ref=("_lon", mode_or_first),
                    n_records=("station_id", "size"))
               .reset_index())

pairs = []
coords = station_ref.dropna(subset=["lat_ref", "lon_ref"]).reset_index(drop=True)
for i in range(len(coords)):
    for j in range(i + 1, len(coords)):
        a, b = coords.iloc[i], coords.iloc[j]
        if a.lat_ref == b.lat_ref and a.lon_ref == b.lon_ref:
            pairs.append([a.station_id, b.station_id, "duplicate_exact", 0.0])
        else:
            d = haversine(a.lat_ref, a.lon_ref, b.lat_ref, b.lon_ref)
            if d < NEAR_DUP_THRESHOLD_M:
                pairs.append([a.station_id, b.station_id, "duplicate_near", d])

dup_df = pd.DataFrame(pairs, columns=["station_id_A", "station_id_B", "dup_flag", "distance_m"])
dup_df.to_csv(OUT_DUP_PAIRS, index=False)

# Map duplicate info back to stations
dup_map = {}
for _, r in dup_df.iterrows():
    dup_map.setdefault(r.station_id_A, []).append(f"{r.dup_flag}:{r.station_id_B}@{int(r.distance_m)}m")
    dup_map.setdefault(r.station_id_B, []).append(f"{r.dup_flag}:{r.station_id_A}@{int(r.distance_m)}m")

station_ref["dup_neighbors"] = station_ref["station_id"].map(lambda x: ";".join(dup_map.get(x, [])))
station_ref["has_dup"] = np.where(station_ref["dup_neighbors"] != "", "YES", "NO")

# ----------------- Final Status Determination -----------------
def determine_status(sid):
    station_issues = details_df[details_df["station_id"] == sid]
    if not station_issues[station_issues["step"].isin(["Step1", "Step2"])].empty:
        return "FAIL"
    if not station_issues[station_issues["step"] == "Step3"].empty or sid in dup_map:
        return "REVIEW"
    return "PASS"

station_ref["qc_status"] = station_ref["station_id"].apply(determine_status)

# Final report formatting
station_out = station_ref[["station_id", "lat_ref", "lon_ref", "n_records", "has_dup", "dup_neighbors", "qc_status"]]
station_out = station_out.sort_values(["qc_status", "station_id"])
station_out.to_csv(OUT_REPORT_STATION, index=False)

print(f"[+] QC Process Completed Successfully.")
print(f"    - Station Report: {OUT_REPORT_STATION}")
print(f"    - Issue Details:  {OUT_REPORT_DETAILS}")
print(f"    - Duplicate List: {OUT_DUP_PAIRS}")

Saved:
 - D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-1-1-station_coordinate_qc_report.csv
 - D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-1-2-station_coordinate_qc_details.csv
 - D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-1-3-station_duplicate_pairs.csv


In [ ]:
#2.3 Repetition

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# ==================== CONFIGURATION ====================
# Using relative paths for GitHub portability
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()

# Input: Files that passed the Physical Limit check
INPUT_GLOB = os.path.join(BASE_DIR, "data", "physical_limit_passed", "*.csv")

# Output: Reports and Summary
REPORT_DIR = os.path.join(BASE_DIR, "output", "repeated_checks")
os.makedirs(REPORT_DIR, exist_ok=True)

OUT_RUNS    = os.path.join(REPORT_DIR, "constant_runs_flagged.csv")
OUT_DELETED = os.path.join(REPORT_DIR, "constant_deleted_records.csv")
OUT_SUMMARY = os.path.join(REPORT_DIR, "constant_summary_by_file.csv")

# Output: Cleaned CSV files (Daily)
CLEANED_DIR = os.path.join(BASE_DIR, "data_cleaned")
os.makedirs(CLEANED_DIR, exist_ok=True)

# Validation Columns
DATE_COL = "date"
RAIN_COL = "rain(mm)"
ID_COLS  = ["station_id", "station_name"]

# Thresholds for Constant Runs
THRESH_VAL = 10.0           # Rainfall must be > 10.0 mm/day to be flagged
MIN_RUNLEN = 5              # Must repeat for >= 5 consecutive days
EPS        = 0.0            # Precision for equality (0.0 means exact match)
DELETE_KEEP_FIRST = False   # If True, keeps the first day of the run and deletes the rest
# =======================================================

# 1) Load and merge all files
files = sorted(glob.glob(INPUT_GLOB))
if not files:
    print(f"[-] No files matched the pattern: {INPUT_GLOB}")
    exit()

print(f"[+] Processing {len(files)} files...")

dfs = []
for f in files:
    df = pd.read_csv(f)
    # Check for required columns
    for c in [DATE_COL, RAIN_COL, "station_id"]:
        if c not in df.columns:
            print(f"[!] Skipping {os.path.basename(f)}: Missing column {c}")
            continue
    df["_source_file"] = os.path.basename(f)
    dfs.append(df)

if not dfs:
    exit()

all_df = pd.concat(dfs, ignore_index=True)

# 2) Data Pre-processing
all_df[DATE_COL] = pd.to_datetime(all_df[DATE_COL], errors="coerce")
all_df["_R"] = pd.to_numeric(all_df[RAIN_COL], errors="coerce")

for c in ID_COLS:
    if c not in all_df.columns:
        all_df[c] = np.nan

# 3) Sort by Station and Date
all_df = all_df.sort_values(["station_id", DATE_COL]).reset_index(drop=True)

# 4) Identify Consecutive "Same Value" Runs
all_df["_prev_station"] = all_df["station_id"].shift(1)
all_df["_prev_date"]    = all_df[DATE_COL].shift(1)
all_df["_prev_R"]       = all_df["_R"].shift(1)

same_station   = all_df["station_id"].eq(all_df["_prev_station"])
is_consecutive = (all_df[DATE_COL] - all_df["_prev_date"]).dt.days.eq(1)
same_value     = (all_df["_R"].notna() & all_df["_prev_R"].notna() & (all_df["_R"] - all_df["_prev_R"]).abs() <= EPS)

# Start a new run ID if station changes, date is not consecutive, or value changes
new_run = (~same_station) | (~is_consecutive) | (~same_value)
all_df["_run_id"] = new_run.cumsum()

# 5) Aggregate Run Statistics
runs = (all_df.groupby(["station_id", "_run_id"])
        .agg(run_len=("_R", "size"),
             run_value=("_R", "first"),
             start_date=(DATE_COL, "first"),
             end_date=(DATE_COL, "last"),
             source_files=("_source_file", lambda x: ",".join(sorted(set(x)))))
        .reset_index())

# Filter runs meeting the flagging criteria
flag_runs = runs[(runs["run_value"] > THRESH_VAL) & (runs["run_len"] >= MIN_RUNLEN)].copy()
flag_runs["rule"]   = f"Rain > {THRESH_VAL} & Consecutive Days >= {MIN_RUNLEN}"
flag_runs["action"] = "DELETE"

# 6) Mark rows for deletion
all_df["_delete_flag"] = False

if not flag_runs.empty:
    flag_keys = set(zip(flag_runs["station_id"], flag_runs["_run_id"]))
    # Vectorized flag check
    in_flag_run = all_df.apply(lambda r: (r["station_id"], r["_run_id"]) in flag_keys, axis=1)

    if DELETE_KEEP_FIRST:
        # Keep the 1st record, delete 2nd..N
        all_df["_is_first_in_run"] = ~all_df.duplicated(subset=["station_id", "_run_id"])
        del_mask = in_flag_run & (~all_df["_is_first_in_run"])
    else:
        # Delete the entire run
        del_mask = in_flag_run

    all_df.loc[del_mask, "_delete_flag"] = True

# 7) Save Flagged Runs and Deleted Records Reports
flag_runs.sort_values(["station_id", "start_date"]).to_csv(OUT_RUNS, index=False)

deleted_rows = all_df[all_df["_delete_flag"]].copy()
keep_cols = [c for c in all_df.columns if not c.startswith("_")] + ["_delete_flag"]
deleted_rows[keep_cols].to_csv(OUT_DELETED, index=False)

# 8) Save Cleaned Daily Files and Generate Summary
summary = []
for f in files:
    base_name = os.path.basename(f)
    subset = all_df[all_df["_source_file"] == base_name].copy()
    
    total_records = len(subset)
    deleted_count = int(subset["_delete_flag"].sum())

    # Write Cleaned File
    cleaned_df = subset[~subset["_delete_flag"]].copy()
    temp_cols = [c for c in cleaned_df.columns if c.startswith("_")]
    cleaned_df = cleaned_df.drop(columns=temp_cols)
    cleaned_df.to_csv(os.path.join(CLEANED_DIR, base_name), index=False)

    summary.append({
        "source_file": base_name,
        "total_records": total_records,
        "deleted_records": deleted_count,
        "kept_records": total_records - deleted_count
    })

pd.DataFrame(summary).sort_values("source_file").to_csv(OUT_SUMMARY, index=False)

print(f"\n[+] Processing Completed.")
print(f"    - Flagged Runs Report: {OUT_RUNS}")
print(f"    - Deleted Records Log: {OUT_DELETED}")
print(f"    - Execution Summary:   {OUT_SUMMARY}")
print(f"    - Cleaned Daily Files Saved to: {CLEANED_DIR}/")

In [ ]:
#3.Data Cleansing___Physical Limit Check

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# ==================== CONFIGURATION ====================
# Using relative paths for GitHub portability
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()

# Input directory (where your raw CSV files are located)
INPUT_FOLDER = os.path.join(BASE_DIR, "data", "raw")
RAIN_COL     = "rain(mm)"

# Output: Reports and Summary
REPORT_DIR = os.path.join(BASE_DIR, "output", "physical_checks")
os.makedirs(REPORT_DIR, exist_ok=True)

OUT_DETAILS = os.path.join(REPORT_DIR, "physical_limit_details.csv")
OUT_SUMMARY = os.path.join(REPORT_DIR, "physical_limit_summary.csv")

# Output: Filtered files (DELETE rows removed)
CLEANED_DIR = os.path.join(BASE_DIR, "data_cleaned")
os.makedirs(CLEANED_DIR, exist_ok=True)

# Threshold Criteria
REVIEW_MIN = 229.0   # Values >= 229.0 will be flagged for REVIEW
HARD_CAP   = 742.0   # Values > 742.0 will be marked for DELETE
# =======================================================

files = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.csv")))

if not files:
    print(f"[-] No CSV files found in: {INPUT_FOLDER}")
    exit()

print(f"[+] Starting Physical Limit Check for {len(files)} files...")

detail_rows = []
summary_rows = []

for f in files:
    file_name = os.path.basename(f)
    try:
        df = pd.read_csv(f)
        
        if RAIN_COL not in df.columns:
            print(f"[!] Skipping {file_name}: Missing column '{RAIN_COL}'")
            continue

        # Convert rainfall to numeric (non-numeric becomes NaN)
        df["_R"] = pd.to_numeric(df[RAIN_COL], errors="coerce")
        df["_source_file"] = file_name

        # Initializing QC columns
        df["_qc_action"] = "KEEP"  # Options: KEEP / REVIEW / DELETE
        df["_qc_reason"] = ""

        # 1) Negative value check -> DELETE
        mask_neg = df["_R"] < 0
        df.loc[mask_neg, "_qc_action"] = "DELETE"
        df.loc[mask_neg, "_qc_reason"] = "negative_value"

        # 2) Physical limit check (> 742 mm/day) -> DELETE
        mask_gt_hard = df["_R"] > HARD_CAP
        df.loc[mask_gt_hard, "_qc_action"] = "DELETE"
        df.loc[mask_gt_hard, "_qc_reason"] = f"greater_than_{HARD_CAP}_mm_day"

        # 3) Plausible range check (229–742 mm/day) -> REVIEW (if not already DELETE)
        mask_review = (df["_R"] >= REVIEW_MIN) & (df["_R"] <= HARD_CAP) & (df["_qc_action"] == "KEEP")
        df.loc[mask_review, "_qc_action"] = "REVIEW"
        df.loc[mask_review, "_qc_reason"] = f"in_range_{REVIEW_MIN}_to_{HARD_CAP}"

        # Collect detailed records for flagged data (REVIEW or DELETE)
        flagged = df[df["_qc_action"].isin(["REVIEW", "DELETE"])].copy()
        if not flagged.empty:
            detail_rows.append(flagged)

        # File-level summary
        summary_rows.append({
            "source_file": file_name,
            "total_records": len(df),
            "deleted_negative": int(mask_neg.sum()),
            "deleted_gt_threshold": int(mask_gt_hard.sum()),
            "review_required": int(mask_review.sum()),
            "kept_records": int((df["_qc_action"] == "KEEP").sum())
        })

        # Save Cleaned File (Keep REVIEW/KEEP, drop DELETE)
        cleaned_df = df[df["_qc_action"] != "DELETE"].copy()
        
        # Remove helper columns but keep QC flags for tracking
        cleaned_df = cleaned_df.drop(columns=["_R"])
        cleaned_df.to_csv(os.path.join(CLEANED_DIR, file_name), index=False)

    except Exception as e:
        print(f"[!] Error processing {file_name}: {e}")

# Save detailed report if any issues were found
if detail_rows:
    pd.concat(detail_rows, ignore_index=True).to_csv(OUT_DETAILS, index=False)
else:
    pd.DataFrame(columns=["status"]).to_csv(OUT_DETAILS, index=False)
    with open(OUT_DETAILS, 'w') as f_out:
        f_out.write("status\nno_flags_detected")

# Save overall summary
pd.DataFrame(summary_rows).to_csv(OUT_SUMMARY, index=False)

print(f"\n[+] QC Process Completed Successfully.")
print(f"    - Issue Details:   {OUT_DETAILS}")
print(f"    - QC Summary:      {OUT_SUMMARY}")
print(f"    - Cleaned Files:   {CLEANED_DIR}/")